# GLAAM Address Matching- All-Splink Pipeline (Exp2)

**all-Splink** matching pipeline against the LDC
commercial premises dataset using OS NGD canonical addresses.

### Pipeline summary

| Setting | Value |
|---------|-------|
| **Matching stage** | `SplinkStage` only (no deterministic stages) |
| **match_weight threshold** | 10 |
| **Canonical data** | 9 NGD CSV types (non-addressable objects excluded) |
| **Postcode strategy** | Full postcodes in address keys |
| **Pre-cleaning** | Residential filtering, OS punctuation stripping, dedup |

### Outputs

- `full_results_exp2.csv` — full matching results (all methods, all candidates)
- `exp2_results_reformatted.csv` — reformatted

## 1. Imports & configuration

In [2]:
import multiprocessing
import os
import tempfile
import time
import zipfile

import duckdb
import numpy as np
import pandas as pd
import psutil
from thefuzz import fuzz

from uk_address_matcher import AddressMatcher, SplinkStage

DATA_DIR     = 'data'
NGD_DATA_DIR = os.path.join(DATA_DIR, 'ngd')
LDC_FILE     = os.path.join(DATA_DIR, 'ldc', 'ldc_history_2025-12-31.csv')

cores = multiprocessing.cpu_count()
container_memory = os.environ.get('DUCKDB_MEMORY_LIMIT', '')
temp_dir = tempfile.gettempdir()
total_gb = psutil.virtual_memory().total // (1024 ** 3)
mem_limit = int(total_gb * 0.9)

print(f'Cores: {cores}, Memory: {container_memory or f"{mem_limit}GB"}, Temp: {temp_dir}')

Cores: 8, Memory: 9GB, Temp: /tmp


## 2. Data-loading helpers

In [3]:
_DEFAULT_COLS = [
    'uprn', 'organisationname', 'subname', 'name', 'number',
    'streetname', 'locality', 'townname',
    'primaryclassificationdescription', 'postcode',
    'fulladdress', 'latitude', 'longitude',
]
_ALT_COLS = [
    'uprn', 'alternatesubname', 'alternatename', 'alternatenumber',
    'streetname', 'locality', 'townname', 'postcode',
    'fulladdress', 'addressstatus',
]
_PSTL_COLS = [
    'uprn', 'organisationname', 'subbuildingname', 'buildingname',
    'buildingnumber', 'thoroughfare', 'dependentlocality',
    'posttown', 'postcode',
]


def _number_str(series):
    return series.astype(str).replace('<NA>', '').replace('nan', '')


def _load_ngd_csv(path):
    fname = os.path.basename(path)
    if '_altadd' in fname:
        df = pd.read_csv(path, usecols=_ALT_COLS)
        df = df.rename(columns={
            'alternatesubname': 'subname', 'alternatename': 'name',
            'alternatenumber': 'number', 'addressstatus': 'source',
        })
        df['source'] = 'alternative'
    elif '_pstladd' in fname:
        df = pd.read_csv(path, usecols=_PSTL_COLS)
        df = df.rename(columns={
            'subbuildingname': 'subname', 'buildingname': 'name',
            'buildingnumber': 'number', 'thoroughfare': 'streetname',
            'dependentlocality': 'locality', 'posttown': 'townname',
        })
        df['source'] = 'postal'
        df['number'] = pd.array(df['number'], dtype='Int64')
    else:
        df = pd.read_csv(path, usecols=_DEFAULT_COLS)
        df['source'] = 'main'
    df['number_str'] = _number_str(df['number'])
    df['type'] = fname.split('_')[2].split('.')[0]
    return df


def _join(*parts):
    return ', '.join(
        str(p).upper() for p in parts
        if pd.notna(p) and str(p) not in ('', 'nan', 'NAN')
    )


def build_canonical_key(df, fields, uid_prefix='c'):
    subset = df[['uprn'] + fields].copy()
    subset['address_c'] = [_join(*row) for row in subset[fields].itertuples(index=False)]
    result = subset[['uprn', 'address_c']].drop_duplicates().sort_values('uprn')
    result = result.reset_index(drop=True).reset_index().rename(columns={'index': f'uid_{uid_prefix}'})
    result[f'uid_{uid_prefix}'] = result[f'uid_{uid_prefix}'].astype(str)
    return result


def build_messy_key(df, fields, uid_prefix='m'):
    subset = df[['uprn', 'premises_id'] + fields].copy()
    subset['address_m'] = [_join(*row) for row in subset[fields].itertuples(index=False)]
    result = subset[['uprn', 'premises_id', 'address_m']].drop_duplicates().sort_values('premises_id')
    result = result.reset_index(drop=True).reset_index().rename(columns={'index': f'uid_{uid_prefix}'})
    result[f'uid_{uid_prefix}'] = result[f'uid_{uid_prefix}'].astype(str)
    return result


print('Data-loading helpers defined.')

Data-loading helpers defined.


## 3. Load OS NGD canonical data

In [4]:
zip_list = [
    f for f in os.listdir(NGD_DATA_DIR)
    if f.endswith('.zip') and 'streetaddress' not in f
]
for zf in zip_list:
    zip_path = os.path.join(NGD_DATA_DIR, zf)
    with zipfile.ZipFile(zip_path) as z:
        to_extract = [m for m in z.namelist() if 'rltenty.csv' not in m and 'otrclass.csv' not in m]
        for member in to_extract:
            dest = os.path.join(NGD_DATA_DIR, member)
            if not os.path.exists(dest):
                z.extract(member, NGD_DATA_DIR)
print(f'ZIPs processed: {len(zip_list)}')

csv_files = [os.path.join(NGD_DATA_DIR, f) for f in os.listdir(NGD_DATA_DIR) if f.endswith('.csv')]
print(f'Loading {len(csv_files)} CSV files...')
os_df = pd.concat([_load_ngd_csv(f) for f in csv_files], axis=0)
os_df = os_df.replace(pd.NA, np.nan)
print(f'Raw OS rows: {len(os_df):,}')

ZIPs processed: 0
Loading 9 CSV files...
Raw OS rows: 10,887,288


## 4. Clean OS data

- Filter out residential addresses
- Deduplicate
- Strip punctuation from OS fields (apostrophes, periods)

In [5]:
os_df = os_df.sort_values(['uprn', 'primaryclassificationdescription'])
os_df['primaryclassificationdescription'] = (
    os_df.groupby('uprn', sort=False)['primaryclassificationdescription'].ffill()
)
os_df = os_df[os_df['primaryclassificationdescription'] != 'Residential']
os_df['latitude']  = os_df.groupby('uprn', sort=False)['latitude'].ffill()
os_df['longitude'] = os_df.groupby('uprn', sort=False)['longitude'].ffill()
os_df = os_df.drop_duplicates(
    subset=['uprn', 'organisationname', 'subname', 'name',
            'number_str', 'streetname', 'townname', 'postcode'],
    keep='first',
)
os_df['postcode_sector'] = os_df['postcode'].str[:-2]
for _col in ['organisationname', 'subname', 'name', 'streetname']:
    os_df[_col] = os_df[_col].str.replace("'", '', regex=False).str.replace('.', '', regex=False)

print(f'Commercial rows after cleaning: {len(os_df):,}')

Commercial rows after cleaning: 1,163,429


## 5. Load LDC messy data

In [6]:
ldc_raw = pd.read_csv(LDC_FILE)
ldc_raw['address'] = (
    ldc_raw['address']
    .str.replace("'", '', regex=False)
    .str.replace('.', '', regex=False)
)
print(f'LDC raw rows: {len(ldc_raw):,}')

ldc_split = ldc_raw['address'].str.split(', ', n=10, expand=True)
for _ in range(ldc_split.shape[1]):
    no_postcode = ldc_split.index[ldc_split[ldc_split.shape[1] - 1].isna()].tolist()
    ldc_split.iloc[no_postcode] = ldc_split.iloc[no_postcode].shift(periods=1, axis=1)

n = ldc_split.shape[1]
ldc_df = ldc_split.drop(columns=[n - 3, n - 2])
ldc_df['organisationname'] = ldc_raw['tenant']
ldc_df['uprn']             = ldc_raw['uprn_id']
ldc_df['premises_id']      = ldc_raw['premises_id']
ldc_df['postcode']         = ldc_df[n - 1]
ldc_df['postcode_sector']  = ldc_df['postcode'].str[:-2]
ldc_df['streetname']       = ldc_df[n - 4]
ldc_df = ldc_df.drop_duplicates()

print(f'LDC after splitting: {len(ldc_df):,} rows')

LDC raw rows: 347,378
LDC after splitting: 280,423 rows


## 6. Build canonical & messy address keys

In [7]:
canonical_a = build_canonical_key(os_df, ['organisationname', 'streetname', 'postcode'], 'ca')
canonical_b = build_canonical_key(os_df, ['subname', 'number_str', 'streetname', 'postcode'], 'cb')
canonical_c = build_canonical_key(os_df, ['name', 'number_str', 'streetname', 'postcode'], 'cc')
canonical_d = build_canonical_key(os_df, ['name', 'subname', 'streetname', 'postcode'], 'cd')
canonical_e = build_canonical_key(os_df, ['number_str', 'streetname', 'postcode'], 'ce')
canonical_f = build_canonical_key(os_df, ['name', 'subname', 'number_str', 'streetname', 'postcode'], 'cf')

for label, df in [('A', canonical_a), ('B', canonical_b), ('C', canonical_c),
                  ('D', canonical_d), ('E', canonical_e), ('F', canonical_f)]:
    print(f'Canonical {label}: {len(df):,} rows')

Canonical A: 1,052,921 rows
Canonical B: 1,027,922 rows
Canonical C: 1,008,306 rows
Canonical D: 1,062,190 rows
Canonical E: 974,743 rows
Canonical F: 1,051,097 rows


In [8]:
_addr_token_cols = [c for c in ldc_df.columns if isinstance(c, int) and c <= n - 5]

messy_a = build_messy_key(ldc_df, ['organisationname', 'streetname', 'postcode'], 'ma')
messy_b = build_messy_key(ldc_df, _addr_token_cols + ['streetname', 'postcode'], 'mb')

print(f'Messy a: {len(messy_a):,} rows')
print(f'Messy b: {len(messy_b):,} rows')

pair_list_6 = [
    (messy_a, canonical_a, 'a'),
    (messy_b, canonical_b, 'b'),
    (messy_b, canonical_c, 'c'),
    (messy_b, canonical_d, 'd'),
    (messy_b, canonical_e, 'e'),
    (messy_b, canonical_f, 'f'),
]
print(f'Pair list: {len(pair_list_6)} rounds ready')

Messy a: 280,329 rows
Messy b: 161,796 rows
Pair list: 6 rounds ready


In [57]:
messy_a

,uid_ma,uprn,premises_id,address_m
0,0,NaN,50000000,"VACANT PROPERTY, WESTBOURNE PARK ROAD, W11 1EH"
1,1,NaN,50000000,"SAMANTHA CUSICK, WESTBOURNE PARK ROAD, W11 1EH"
2,2,NaN,50000000,"10500, WESTBOURNE PARK ROAD, W11 1EH"
3,3,NaN,50000001,"BLINK BROW BAR, LEDBURY ROAD, W11 2AA"
4,4,NaN,50000001,"VACANT PROPERTY, LEDBURY ROAD, W11 2AA"
...,...,...,...,...
280324,280324,NaN,53132444,"TACO BELL, HIGH STREET, UB8 1GB"
280325,280325,NaN,53132445,"TK MAXX, HIGH STREET, UB8 1GB"
280326,280326,NaN,53132446,"FRESH COLLECTION, OXFORD STREET, W1D 2LP"
280327,280327,NaN,53132465,"BRITISH HEART FOUNDATION FURNITURE & ELECTRICAL, PANTILE..."


## 7. Matching helpers

In [9]:
def make_duckdb_con():
    c = duckdb.connect(database=':memory:')
    c.execute(f'PRAGMA threads={cores}')
    if container_memory:
        c.execute(f"PRAGMA memory_limit='{container_memory}'")
    else:
        c.execute(f"PRAGMA memory_limit='{mem_limit}GB'")
    c.execute(f"SET temp_directory='{temp_dir}'")
    return c


def run_matching_loop(pair_list, stages, label='experiment'):
    """Run the full 6-round matching loop with a given stage config."""
    import tempfile as _tmpmod
    _TMP_MESSY  = os.path.join(_tmpmod.gettempdir(), '_exp_messy.parquet')
    _TMP_CANON  = os.path.join(_tmpmod.gettempdir(), '_exp_canon.parquet')

    all_results = []
    t0 = time.time()

    for i, (df_messy_pd, df_canon_pd, method) in enumerate(pair_list):
        uid_m_col = [c for c in df_messy_pd.columns if c.startswith('uid_')][0]
        uid_c_col = [c for c in df_canon_pd.columns if c.startswith('uid_')][0]

        input_df = df_messy_pd.rename(
            columns={uid_m_col: 'unique_id', 'address_m': 'address_concat'}
        )[['unique_id', 'address_concat']].copy()
        input_df['postcode'] = input_df['address_concat'].str.rsplit(', ', n=1).str[-1]

        ref_df = df_canon_pd.rename(
            columns={uid_c_col: 'unique_id', 'address_c': 'address_concat'}
        )[['unique_id', 'address_concat']].copy()
        ref_df['postcode'] = ref_df['address_concat'].str.rsplit(', ', n=1).str[-1]

        input_df.to_parquet(_TMP_MESSY)
        ref_df.to_parquet(_TMP_CANON)

        round_con = make_duckdb_con()
        pq_input = round_con.read_parquet(_TMP_MESSY)
        pq_ref   = round_con.read_parquet(_TMP_CANON)

        print(f'  [{label}] Round {i} (method {method!r}) — '
              f'{df_messy_pd.shape[0]:,} x {df_canon_pd.shape[0]:,}')
        t_round = time.time()

        matcher = AddressMatcher(
            canonical_addresses=pq_ref,
            addresses_to_match=pq_input,
            con=round_con,
            stages=stages,
        )
        match_result = matcher.match()
        df_result = match_result.matches().df()
        df_result = df_result[df_result['resolved_canonical_id'].notna()]
        df_result = df_result.rename(columns={
            'unique_id': uid_m_col,
            'resolved_canonical_id': uid_c_col,
            'original_address_concat': 'address_m',
            'original_address_concat_canonical': 'address_c',
        })

        if df_result.empty:
            print(f'    No matches — skipping.')
            round_con.close()
            continue

        df_result['fuzz_similarity'] = df_result.apply(
            lambda r: fuzz.ratio(
                str(r.get('address_m','')).upper(),
                str(r.get('address_c','')).upper()
            ), axis=1
        )

        combined_df = df_messy_pd.merge(
            df_result[[uid_m_col, uid_c_col, 'match_weight',
                       'distinguishability', 'fuzz_similarity']],
            on=uid_m_col, how='left',
        ).merge(
            df_canon_pd, on=uid_c_col, how='left', suffixes=['_m', '_c'],
        ).sort_values(['premises_id', 'fuzz_similarity'], ascending=[True, False])
        combined_df['method'] = method
        all_results.append(combined_df)
        matched_count = combined_df[uid_c_col].notna().sum()
        round_con.close()
        print(f'    -> {matched_count:,} matched in {time.time()-t_round:.1f}s')

    elapsed = time.time() - t0
    print(f'  [{label}] All rounds done in {elapsed:.1f}s')

    if not all_results:
        return pd.DataFrame()

    full = pd.concat(all_results, ignore_index=True)
    if 'uprn' in full.columns and 'uprn_c' not in full.columns:
        full = full.rename(columns={'uprn': 'uprn_c'})
    full['premises_id'] = full['premises_id'].astype(str)
    if 'uprn_c' in full.columns:
        full['uprn_c'] = (full['uprn_c'].astype(str)
                          .str.replace(r'\.0$', '', regex=True))
    return full


print('Matching helpers defined.')

Matching helpers defined.


## 8. Run all-Splink pipeline

Single `SplinkStage` with `match_weight >= 10` threshold.
Every matched row receives a `match_weight` and `distinguishability` score.

In [10]:
STAGES = [
    SplinkStage(
        predict_threshold_match_weight=-20,
        final_match_weight_threshold=10,
        include_full_postcode_block=False,
        retain_intermediate_calculation_columns=False,
    ),
]

print('Running all-Splink pipeline (6 address-key rounds) ...')
print()
results = run_matching_loop(pair_list_6, STAGES, label='Exp2')
results.to_csv('full_result_exp2.csv', index=False)
print(f'\nSaved {len(results):,} rows to full_result_exp2.csv')

matched = results[results['uprn_c'].notna() & (results['uprn_c'] != 'nan')]
hit_premises = matched.groupby('premises_id').first().reset_index()
total = results['premises_id'].nunique()
print(f'Unique premises: {total:,}')
print(f'Matched premises (hit): {len(hit_premises):,} ({len(hit_premises)/total:.1%})')

Running all-Splink pipeline (6 address-key rounds) ...

  [Exp2] Round 0 (method 'a') — 280,329 x 1,052,921
    -> 101,336 matched in 97.3s
  [Exp2] Round 1 (method 'b') — 161,796 x 1,027,922
    -> 138,412 matched in 86.9s
  [Exp2] Round 2 (method 'c') — 161,796 x 1,008,306
    -> 135,633 matched in 90.2s
  [Exp2] Round 3 (method 'd') — 161,796 x 1,062,190
    -> 46,983 matched in 80.6s
  [Exp2] Round 4 (method 'e') — 161,796 x 974,743
    -> 133,796 matched in 79.1s
  [Exp2] Round 5 (method 'f') — 161,796 x 1,051,097
    -> 140,807 matched in 100.8s
  [Exp2] All rounds done in 558.0s

Saved 1,089,309 rows to full_result_exp2.csv
Unique premises: 127,161
Matched premises (hit): 118,464 (93.2%)


## 9. Reformat results

In [10]:
TARGET_COLUMNS = [
    'uid_m', 'uprn_m', 'premises_id', 'address_m', 'uid_c',
    'match_weight', 'distinguishability', 'fuzz_similarity',
    'uprn_c', 'address_c', 'method',
]

exp2_share = results.rename(columns={'uid_ma': 'uid_m', 'uid_ca': 'uid_c'})

drop_cols = [c for c in ['uid_mb', 'uid_cb', 'uid_cc', 'uid_cd', 'uid_ce', 'uid_cf']
             if c in exp2_share.columns]
exp2_share = exp2_share.drop(columns=drop_cols)

final_cols = [c for c in TARGET_COLUMNS if c in exp2_share.columns]
exp2_share = exp2_share[final_cols]

out_file = 'exp2_results_reformatted.csv'
exp2_share.to_csv(out_file, index=False)

print(f'Saved: {out_file}')
print(f'Columns: {list(exp2_share.columns)}')
print(f'Rows: {len(exp2_share):,}')
print(f'\nFirst 5 matched rows:')
print(exp2_share[exp2_share['uprn_c'].notna()].head().to_string(index=False, max_colwidth=40))

Saved: exp2_results_reformatted.csv
Columns: ['uid_m', 'uprn_m', 'premises_id', 'address_m', 'uid_c', 'match_weight', 'distinguishability', 'fuzz_similarity', 'uprn_c', 'address_c', 'method']
Rows: 1,089,309
Column order matches target: True

First 5 matched rows:
uid_m  uprn_m premises_id                                address_m  uid_c  match_weight  distinguishability  fuzz_similarity    uprn_c                                address_c method
    1     NaN    50000000 SAMANTHA CUSICK, WESTBOURNE PARK ROAD... 303117     32.496069           39.128819             93.0 217092023 SAMANTHA CUSICK LONDON, WESTBOURNE PA...      a
    0     NaN    50000000 VACANT PROPERTY, WESTBOURNE PARK ROAD...    NaN           NaN                 NaN              NaN       nan                                      NaN      a
    2     NaN    50000000     10500, WESTBOURNE PARK ROAD, W11 1EH    NaN           NaN                 NaN              NaN       nan                                      NaN      a
   

## 10. Address uniqueness scores

Derive a per-address uniqueness score from Splink's token frequencies.
For each messy address, we tokenize it, look up each token's `rel_freq` in the
TF table (derived from canonical data), and aggregate into a single score.

In [11]:
# token frequency table from canonical data 

from uk_address_matcher.cleaning.chunking_strategies import derive_term_frequencies_table

tf_con = make_duckdb_con()

canonical_full = os_df[['uprn', 'organisationname', 'subname', 'name',
                        'number_str', 'streetname', 'townname', 'postcode']].copy()
canonical_full['address_concat'] = [
    _join(row.organisationname, row.subname, row.name, row.number_str,
          row.streetname, row.townname, row.postcode)
    for row in canonical_full.itertuples()
]
canonical_full['unique_id'] = range(len(canonical_full))
canonical_full['postcode'] = canonical_full['postcode'].fillna('')

canon_rel = tf_con.from_df(canonical_full[['unique_id', 'address_concat', 'postcode']])

print('Deriving token frequencies from canonical data...')
tf_table = derive_term_frequencies_table(canon_rel, con=tf_con)
tf_df = tf_table.df()

print(f'Token frequency table: {len(tf_df):,} tokens')
print(f'Frequency range: {tf_df["rel_freq"].min():.2e} — {tf_df["rel_freq"].max():.3f}')
print(f'\nSample (rarest 10):')
print(tf_df.nsmallest(10, 'rel_freq').to_string(index=False))
print(f'\nSample (most common 10):')
print(tf_df.nlargest(10, 'rel_freq').to_string(index=False))

Deriving token frequencies from canonical data...
Token frequency table: 124,931 tokens
Frequency range: 1.26e-07 — 0.092

Sample (rarest 10):
       token     rel_freq
      ONIONS 1.259398e-07
FITZCLARENCE 1.259398e-07
   GRANSOLAR 1.259398e-07
     MULLANE 1.259398e-07
   WAYBEARDS 1.259398e-07
  1432-1432A 1.259398e-07
    40125008 1.259398e-07
    EASKDALE 1.259398e-07
      03C116 1.259398e-07
       2-066 1.259398e-07

Sample (most common 10):
  token  rel_freq
 LONDON  0.092405
   ROAD  0.065843
 STREET  0.028970
  FLOOR  0.016701
   UNIT  0.016411
  HOUSE  0.015205
   LANE  0.010715
LIMITED  0.010442
    AND  0.010334
   HIGH  0.008944


In [12]:
# Compute uniqueness scores for all messy addresses 

import re, math

UNKNOWN_FREQ = 5e-5  # same floor as the library uses for unknown tokens

STOP_TOKENS = {
    'ROAD', 'STREET', 'LANE', 'AVENUE', 'DRIVE', 'CLOSE', 'WAY',
    'COURT', 'PLACE', 'GARDENS', 'TERRACE', 'GROVE', 'CRESCENT',
    'LONDON', 'HOUSE', 'PARK', 'THE', 'AND', 'OF',
}

tf_lookup = dict(zip(tf_df['token'], tf_df['rel_freq']))

def compute_uniqueness(address_str):
    if not isinstance(address_str, str) or not address_str.strip():
        return {}

    cleaned = address_str.upper().strip()
    cleaned = re.sub(r"[''.,]", '', cleaned)
    tokens = cleaned.split()

    # Remove postcode tokens (last 1-2 tokens that look like postcode parts)
    pc_pattern = re.compile(r'^[A-Z]{1,2}\d|^\d[A-Z]{2}$')
    content_tokens = []
    for t in tokens:
        if pc_pattern.match(t):
            continue
        if re.match(r'^\d+$', t):
            continue
        content_tokens.append(t)

    if not content_tokens:
        return {'n_tokens': 0, 'min_freq': None, 'mean_freq': None,
                'median_freq': None, 'log_uniqueness': None}

    freqs = [tf_lookup.get(t, UNKNOWN_FREQ) for t in content_tokens]
    freqs_sorted = sorted(freqs)
    n = len(freqs)

    non_stop_freqs = [tf_lookup.get(t, UNKNOWN_FREQ)
                      for t in content_tokens if t not in STOP_TOKENS]

    return {
        'n_tokens': n,
        'min_freq': min(freqs),
        'mean_freq': sum(freqs) / n,
        'median_freq': freqs_sorted[n // 2],
        'log_uniqueness': -sum(math.log(f) for f in freqs),
        'min_freq_non_stop': min(non_stop_freqs) if non_stop_freqs else None,
        'log_uniqueness_non_stop': (
            -sum(math.log(f) for f in non_stop_freqs) if non_stop_freqs else None
        ),
    }

# Build full messy address from ldc_raw (one row per premises)
messy_addresses = ldc_raw[['premises_id', 'address']].drop_duplicates(subset='premises_id')
messy_addresses = messy_addresses.copy()
messy_addresses['premises_id'] = messy_addresses['premises_id'].astype(str)

scores = messy_addresses['address'].apply(compute_uniqueness).apply(pd.Series)
messy_scores = pd.concat([messy_addresses.reset_index(drop=True), scores], axis=1)

print(f'Computed uniqueness scores for {len(messy_scores):,} premises')
print(f'\nScore distributions:')
for col in ['min_freq', 'mean_freq', 'median_freq', 'log_uniqueness',
            'min_freq_non_stop', 'log_uniqueness_non_stop']:
    vals = messy_scores[col].dropna()
    print(f'  {col:25s}  mean={vals.mean():.4f}  median={vals.median():.4f}  '
          f'std={vals.std():.4f}')

Computed uniqueness scores for 162,832 premises

Score distributions:
  min_freq                   mean=0.0000  median=0.0000  std=0.0000
  mean_freq                  mean=0.0349  median=0.0361  std=0.0105
  median_freq                mean=0.0242  median=0.0089  std=0.0272
  log_uniqueness             mean=37.7618  median=33.6824  std=11.4660
  min_freq_non_stop          mean=0.0000  median=0.0000  std=0.0000
  log_uniqueness_non_stop    mean=30.2748  median=26.9419  std=11.4270


In [16]:
# Join uniqueness scores to matching results 

results_df = pd.read_csv('full_results_exp2.csv', dtype={'premises_id': str, 'uprn_c': str})

score_cols = ['premises_id', 'min_freq', 'mean_freq', 'median_freq',
              'log_uniqueness', 'min_freq_non_stop', 'log_uniqueness_non_stop']
results_with_scores = results_df.merge(
    messy_scores[score_cols], on='premises_id', how='left'
)

out_file = 'exp2_results_with_uniqueness.csv'
results_with_scores.to_csv(out_file, index=False)
print(f'Saved: {out_file}')
print(f'Rows: {len(results_with_scores):,}')
print(f'Columns: {list(results_with_scores.columns)}')

# Summary: matched vs unmatched uniqueness
matched = results_with_scores[
    results_with_scores['uprn_c'].notna() & (results_with_scores['uprn_c'] != 'nan')
]
unmatched = results_with_scores[
    results_with_scores['uprn_c'].isna() | (results_with_scores['uprn_c'] == 'nan')
]

best_matched = matched.groupby('premises_id').first().reset_index()
best_unmatched = unmatched.groupby('premises_id').first().reset_index()

print(f'\n{"Metric":30s} {"Matched":>12s} {"Unmatched":>12s}')
print('-' * 56)
for col in ['min_freq', 'log_uniqueness', 'log_uniqueness_non_stop']:
    m_val = best_matched[col].median()
    u_val = best_unmatched[col].median()
    print(f'{col:30s} {m_val:12.4f} {u_val:12.4f}')

print(f'\nPremises: {len(best_matched):,} matched, {len(best_unmatched):,} unmatched')

Saved: exp2_results_with_uniqueness.csv
Rows: 1,089,309
Columns: ['uid_ma', 'uprn_m', 'premises_id', 'address_m', 'uid_ca', 'match_weight', 'distinguishability', 'fuzz_similarity', 'uprn_c', 'address_c', 'method', 'uid_mb', 'uid_cb', 'uid_cc', 'uid_cd', 'uid_ce', 'uid_cf', 'min_freq', 'mean_freq', 'median_freq', 'log_uniqueness', 'min_freq_non_stop', 'log_uniqueness_non_stop']

Metric                              Matched    Unmatched
--------------------------------------------------------
min_freq                             0.0000       0.0000
log_uniqueness                      33.7882      33.7882
log_uniqueness_non_stop             27.1964      27.1981

Premises: 120,093 matched, 115,015 unmatched


In [17]:
# Export TF table and uniqueness scores 

tf_df.to_csv('token_frequencies.csv', index=False)
print(f'Saved: token_frequencies.csv ({len(tf_df):,} tokens)')

messy_scores.to_csv('messy_uniqueness_scores.csv', index=False)
print(f'Saved: messy_uniqueness_scores.csv ({len(messy_scores):,} premises)')

print('\nFiles ready for classifier training:')
print('  - token_frequencies.csv       (global TF lookup)')
print('  - messy_uniqueness_scores.csv (per-premises uniqueness scores)')
print('  - exp2_results_with_uniqueness.csv (matching results + scores joined)')

Saved: token_frequencies.csv (124,931 tokens)
Saved: messy_uniqueness_scores.csv (162,832 premises)

Files ready for classifier training:
  - token_frequencies.csv       (global TF lookup)
  - messy_uniqueness_scores.csv (per-premises uniqueness scores)
  - exp2_results_with_uniqueness.csv (matching results + scores joined)


In [21]:
import numpy as np

tf_lookup = dict(zip(tf_df['token'], tf_df['rel_freq']))

# One row per premises with the raw address
messy = ldc_raw[['premises_id', 'address']].drop_duplicates(subset='premises_id').copy()
messy['premises_id'] = messy['premises_id'].astype(str)

# Tokenize: uppercase, strip punctuation, split on spaces/commas
messy['tokens'] = (
    messy['address']
    .str.upper()
    .str.replace(r"[''.,]", '', regex=True)
    .str.split()
)
messy['freq_array'] = messy['tokens'].apply(
    lambda toks: np.array([tf_lookup.get(t, np.nan) for t in toks]) if isinstance(toks, list) else np.array([])
)


messy['min_freq']    = messy['freq_array'].apply(lambda a: np.nanmin(a) if len(a) > 0 else np.nan)
messy['mean_freq']   = messy['freq_array'].apply(lambda a: np.nanmean(a) if len(a) > 0 else np.nan)
messy['median_freq'] = messy['freq_array'].apply(lambda a: np.nanmedian(a) if len(a) > 0 else np.nan)


pd.set_option('display.max_colwidth', 60)
print(messy[['premises_id', 'address', 'tokens', 'freq_array', 'min_freq', 'mean_freq', 'median_freq']].head(10).to_string(index=False))

# # Save
# messy.to_csv('messy_uniqueness_v2.csv', index=False)
# print(f'\nSaved: messy_uniqueness_v2.csv ({len(messy):,} rows)')

premises_id                                                                                               address                                                                                                           tokens                                                                                                                                                                                                                                                                                     freq_array     min_freq  mean_freq  median_freq
   52054865                                            125-125A, Northcote Road, London, Greater London, SW11 6PS                                                  [125-125A, NORTHCOTE, ROAD, LONDON, GREATER, LONDON, SW11, 6PS]                                                                                                                               [3.778194778534816e-07, 8.286840547586363e-05, 0.0658427263453522, 0.09240545067566716, 1.070488

## Google Geocode API

In [1]:
import requests
import time
import pandas as pd
import numpy as np
GEOCODE_URL = 'https://maps.googleapis.com/maps/api/geocode/json'
def geocode_address(address, api_key, region='gb', country='GB'):
    """ Geocode a single address via Google Geocoding API.
    
    Returns a dict with: status, formatted_address, lat, lng,
    google_postcode, place_id, location_type, and all address components.
    """
    params = {
        'address': address,
        'key': api_key,
        'region': region,
        'components': f'country:{country}',
    }
    
    try:
        resp = requests.get(GEOCODE_URL, params=params, timeout=10)
        data = resp.json()
        print(data)
    except Exception as e:
        return {'status': 'REQUEST_ERROR', 'error': str(e), 'input_address': address}
    
    if data['status'] != 'OK' or not data.get('results'):
        return {'status': data['status'], 'input_address': address}
    
    result = data['results'][0]
    loc = result['geometry']['location']
    
    # Extract address components
    components = {}
    for comp in result.get('address_components', []):
        for t in comp['types']:
            components[t] = comp['long_name']
    
    return {
        'status': 'OK',
        'input_address': address,
        'formatted_address': result.get('formatted_address'),
        'lat': loc['lat'],
        'lng': loc['lng'],
        'google_postcode': components.get('postal_code'),
        'google_street_number': components.get('street_number'),
        'google_route': components.get('route'),
        'google_locality': components.get('postal_town') or components.get('locality'),
        'google_admin_area': components.get('administrative_area_level_2'),
        'place_id': result.get('place_id'),
        'location_type': result['geometry'].get('location_type'),
        'types': ', '.join(result.get('types', [])),
    }

In [2]:
# function to batch process the addresses from df with rate limiting and checkpointing -- geocode
def geocode_batch(df, address_col, api_key, premises_id_col='premises_id',
                  requests_per_second=45, save_every=500, output_file=None):
    """Geocode a DataFrame of addresses with rate limiting and checkpointing.
    
    Args:
        df: DataFrame with addresses to geocode
        address_col: column name containing the address string
        api_key: Google Maps API key
        premises_id_col: column name for unique ID
        requests_per_second: rate limit (Google allows 50/s, leave headroom)
        save_every: checkpoint interval (saves progress to CSV)
        output_file: path for checkpoint CSV (None = no checkpointing)
    
    Returns:
        DataFrame with geocoding results joined to original data
    """
    results = []
    total = len(df)
    delay = 1.0 / requests_per_second
    
    # Load existing progress if checkpoint file exists
    done_ids = set()
    if output_file:
        try:
            existing = pd.read_csv(output_file, dtype={premises_id_col: str})
            done_ids = set(existing[premises_id_col].astype(str))
            results = existing.to_dict('records')
            print(f'Resumed from checkpoint: {len(done_ids):,} already done')
        except FileNotFoundError:
            pass
    
    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        pid = str(row[premises_id_col])
        if pid in done_ids:
            continue
        
        address = row[address_col]
        geo = geocode_address(address, api_key)
        geo[premises_id_col] = pid
        results.append(geo)
        
        # Progress
        done = len(results)
        if done % 100 == 0 or done == total:
            elapsed = time.time() - t0
            rate = done / elapsed if elapsed > 0 else 0
            remaining = (total - done) / rate if rate > 0 else 0
            print(f'  [{done:,}/{total:,}] {rate:.1f} req/s, '
                  f'~{remaining/60:.1f} min remaining', end='\r')
        
        # Checkpoint
        if output_file and done % save_every == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
        
        time.sleep(delay)
    
    print(f'\nDone: {len(results):,} addresses geocoded in {time.time()-t0:.1f}s')
    
    result_df = pd.DataFrame(results)
    if output_file:
        result_df.to_csv(output_file, index=False)
        print(f'Saved: {output_file}')
    
    return result_df

In [6]:
# TEST: single address

API_KEY = '--API--'  
test_result = geocode_address(
    'flat 2, 134A High Street, UB8 2JX',
    api_key=API_KEY
)
for k, v in test_result.items():
    print(f'  {k:25s}: {v}')

{'results': [{'address_components': [{'long_name': '2', 'short_name': '2', 'types': ['subpremise']}, {'long_name': '134a', 'short_name': '134a', 'types': ['street_number']}, {'long_name': 'High Street', 'short_name': 'High St', 'types': ['route']}, {'long_name': 'Uxbridge', 'short_name': 'Uxbridge', 'types': ['postal_town']}, {'long_name': 'Greater London', 'short_name': 'Greater London', 'types': ['administrative_area_level_2', 'political']}, {'long_name': 'England', 'short_name': 'England', 'types': ['administrative_area_level_1', 'political']}, {'long_name': 'United Kingdom', 'short_name': 'GB', 'types': ['country', 'political']}, {'long_name': 'UB8 1JX', 'short_name': 'UB8 1JX', 'types': ['postal_code']}], 'formatted_address': '2, 134a High St, Uxbridge UB8 1JX, UK', 'geometry': {'location': {'lat': 51.5477004, 'lng': -0.4806421999999999}, 'location_type': 'ROOFTOP', 'viewport': {'northeast': {'lat': 51.54902248029151, 'lng': -0.479379919708498}, 'southwest': {'lat': 51.54632451970

In [74]:
import requests
import time
import pandas as pd
import numpy as np

PLACES_URL = 'https://places.googleapis.com/v1/places:searchText'

# Select fields 
FIELD_MASK = ','.join([
    'places.id',
    'places.displayName',
    'places.formattedAddress',
    'places.shortFormattedAddress',
    'places.location',
    'places.addressComponents',
    'places.types',
    'places.primaryType',
])


def places_search(query, api_key, region='gb'):
    """Search for a place using Google Places Text Search (New) API.

    Returns a dict with: status, display_name, formatted_address,
    lat, lng, google_postcode, place_id, types, and address components.
    """
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': FIELD_MASK,
    }
    body = {
        'textQuery': query,
        'languageCode': 'en',
        'regionCode': region.upper(),
    }

    try:
        resp = requests.post(PLACES_URL, headers=headers, json=body, timeout=10)
        data = resp.json()
    except Exception as e:
        return {'status': 'REQUEST_ERROR', 'error': str(e), 'input_query': query}

    if 'error' in data:
        return {
            'status': data['error'].get('status', 'ERROR'),
            'error': data['error'].get('message', ''),
            'input_query': query,
        }

    places = data.get('places', [])
    if not places:
        return {'status': 'ZERO_RESULTS', 'input_query': query}

    place = places[0]
    loc = place.get('location', {})

    # Extract postcode from addressComponents
    postcode = None
    street_number = None
    route = None
    locality = None
    for comp in place.get('addressComponents', []):
        types = comp.get('types', [])
        if 'postal_code' in types:
            postcode = comp.get('longText')
        elif 'street_number' in types:
            street_number = comp.get('longText')
        elif 'route' in types:
            route = comp.get('longText')
        elif 'postal_town' in types:
            locality = comp.get('longText')
        elif 'locality' in types and not locality:
            locality = comp.get('longText')

    return {
        'status': 'OK',
        'input_query': query,
        'place_id': place.get('id'),
        'display_name': place.get('displayName', {}).get('text'),
        'formatted_address': place.get('formattedAddress'),
        'short_address': place.get('shortFormattedAddress'),
        'lat': loc.get('latitude'),
        'lng': loc.get('longitude'),
        'google_postcode': postcode,
        'google_street_number': street_number,
        'google_route': route,
        'google_locality': locality,
        # 'primary_type': place.get('primaryType'),
        # 'types': ', '.join(place.get('types', [])),
        'num_results': len(places),
    }

# function to batch process the addresses from df with rate limiting and checkpointing -- places
def places_batch(df, api_key, orgname_col='organisationname',
                 street_col='streetname', postcode_col='postcode',
                 premises_id_col='premises_id',
                 requests_per_second=45, save_every=500, output_file=None):
    """Batch Places search: builds query from orgname + street + postcode.

    Args:
        df: DataFrame with address columns
        api_key: Google Maps API key
        orgname_col/street_col/postcode_col: column names for query building
        premises_id_col: unique ID column
        requests_per_second: rate limit (leave headroom below 50/s)
        save_every: checkpoint interval
        output_file: path for checkpoint CSV
    """
    results = []
    total = len(df)
    delay = 1.0 / requests_per_second

    done_ids = set()
    if output_file:
        try:
            existing = pd.read_csv(output_file, dtype={premises_id_col: str})
            done_ids = set(existing[premises_id_col].astype(str))
            results = existing.to_dict('records')
            print(f'Resumed from checkpoint: {len(done_ids):,} already done')
        except FileNotFoundError:
            pass

    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        pid = str(row[premises_id_col])
        if pid in done_ids:
            continue

        parts = [
            str(row.get(orgname_col, '') or ''),
            str(row.get(street_col, '') or ''),
            str(row.get(postcode_col, '') or ''),
        ]
        query = ', '.join(p for p in parts if p.strip())

        result = places_search(query, api_key)
        result[premises_id_col] = pid
        results.append(result)

        done = len(results)
        if done % 100 == 0 or done == total:
            elapsed = time.time() - t0
            rate = done / elapsed if elapsed > 0 else 0
            remaining = (total - done) / rate if rate > 0 else 0
            print(f'  [{done:,}/{total:,}] {rate:.1f} req/s, '
                  f'~{remaining/60:.1f} min remaining', end='\r')

        if output_file and done % save_every == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)

        time.sleep(delay)

    print(f'\nDone: {len(results):,} places searched in {time.time()-t0:.1f}s')

    result_df = pd.DataFrame(results)
    if output_file:
        result_df.to_csv(output_file, index=False)
        print(f'Saved: {output_file}')

    return result_df

In [ ]:
# TEST: single search

API_KEY = ''

test = places_search('The Warehouse Sports And Performing Arts Centre, Speranza Street, SE18 1NX', api_key=API_KEY)
for k, v in test.items():
    print(f'  {k:25s}: {v}')

  input_query              : The Warehouse Sports And Performing Arts Centre, Speranza Street, SE18 1NX
  place_id                 : ChIJFba85jKv2EcRVv77dHtD77U
  display_name             : The Warehouse Sports And Performing Arts Centre
  formatted_address        : Speranza St, London SE18 1NX
  short_address            : Speranza St, London
  lat                      : 51.485693399999995
  lng                      : 0.09573419999999999
  google_postcode          : SE18 1NX
  google_street_number     : None
  google_route             : Speranza Street
  google_locality          : London


In [69]:
fail_sample = pd.read_csv('./data/fails_sample.csv')

In [ ]:
places_batch(df = fail_sample, api_key = '', orgname_col='organisationname',
                 street_col='streetname', postcode_col='postcode',
                 premises_id_col='premises_id',
                 requests_per_second=45, save_every=500, output_file='failed_sample_matched.csv')

  [100/100] 2.6 req/s, ~0.0 min remaining
Done: 100 places searched in 39.2s
Saved: failed_sample_matched.csv


,input_query,place_id,display_name,formatted_address,short_address,lat,lng,google_postcode,google_street_number,google_route,google_locality,premises_id,status
0,"Terrace Bar, South Bank, SE1 9PX",ChIJHdub7BEFdkgRvfT25On0IJw,Seventy5th Balcony Bar,"Level 5, Royal Festival Hall, Southbank Centre, London S...","Level 5, Royal Festival Hall, Southbank Centre, Waterloo...",51.505921,-0.117052,SE1 8XX,None,None,London,50020473,NaN
1,"One Beyond, North Square, N9 0HW",ChIJ1bkQjPEedkgRNKPzXk0jpy8,Boots,"28-29 North Square, London N9 0HW","edmonton green shopping centre, 28-29 North Square, London",51.625863,-0.056468,N9 0HW,28-29,North Square,London,50955270,NaN
2,"The Warehouse Sports And Performing Arts Centre, Speranz...",ChIJFba85jKv2EcRVv77dHtD77U,The Warehouse Sports And Performing Arts Centre,"Speranza St, London SE18 1NX","Speranza St, London",51.485693,0.095734,SE18 1NX,None,Speranza Street,London,52845137,NaN
3,"Grab N Go, New Road, TW14 9BG",ChIJN3XQwopxdkgRSE9dRPPdzsY,GRAB N' GO,"49 High St, Stanwell, Staines TW19 7LJ","49 High St, Stanwell, Staines",51.457477,-0.478598,TW19 7LJ,49,High Street,Staines,52441871,NaN
4,"Eastex, Station Road, IG1 4DP",ChIJq6qa0Yqm2EcRJs9FnIldxzk,Eastex,"1-12 Station Rd, Ilford IG1 4DP","1-12 Station Rd, Ilford",51.559147,0.070957,IG1 4DP,1-12,Station Road,Ilford,50694038,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"Special Flights, Tooting High Street, SW17 0SP",ChIJ9RM2-goGdkgR9bUZVrs4T3I,Special Flights,"29 Tooting High St, London SW17 0RJ","29 Tooting High St, London",51.428732,-0.166474,SW17 0RJ,29,Tooting High Street,London,52933983,NaN
96,"Dale Youth Amateur Boxing Club, Maxilla Walk, W10 6NQ",ChIJybQ1ASQQdkgReLj97lWk2Go,Dale Youth Abc,"70 St Marks Rd, London W10 6NP","70 St Marks Rd, London",51.516765,-0.213797,W10 6NP,70,Saint Marks Road,London,52953707,NaN
97,"nan, North Circular Road, NW10 7XP",ChIJAXey5CESdkgRtVnOtmJXdpY,Dean Training,"Rays House, N Circular Rd., London NW10 7XP","Rays House, N Circular Rd., London",51.535834,-0.287395,NW10 7XP,None,North Circular Road,London,52833434,NaN
98,"TH @ 51, Buckingham Gate, SW1E 6BS",ChIJ-y98NTMFdkgRQDy1BedcBZM,Th@51 Restaurant and Bar,"Taj 51, Suites and Residences, 51 Buckingham Gate, Londo...","Suites and Residences, Taj 51, 51 Buckingham Gate, London",51.498665,-0.137494,SW1E 6AF,51,Buckingham Gate,London,50024673,NaN


In [ ]:
fail_sample_fs = pd.read_csv('./data/fails_sample_full_string 1.csv')
results = []
for _, row in fail_sample_fs.iterrows():
    result = places_search(row['biz_input_string'], api_key='')
    result['premises_id'] = str(row['premises_id'])
    results.append(result)
    time.sleep(1/45)

fail_sample_fs_df = pd.DataFrame(results)

In [77]:
fail_sample_fs_df.to_csv('fail_sample_full_string_matched.csv', index=False)